In [1]:
import insightface
from sklearn.metrics.pairwise import cosine_similarity
import os
from insightface.model_zoo import get_model
import numpy as np
from insightface.app import FaceAnalysis
import cv2

model = get_model('buffalo_l/w600k_r50.onnx', download=True)
app = FaceAnalysis(providers=['CUDAExecutionProvider'])
model.prepare(ctx_id=0)


/usr3/graduate/dlgirija/.local/lib/python3.11/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:118: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /usr3/graduate/dlgirija/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /usr3/graduate/dlgirija/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /usr3/graduate/dlgirija/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /usr3/graduate/dlgirija/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionP

In [2]:

# Step 2: Encode reference images
reference_dir = 'output_unmask'
embeddings = []
reference_names = []

for direc in os.listdir(reference_dir):
    for file in os.listdir(os.path.join(reference_dir, direc)):
        if file.lower().endswith(('.jpg', '.jpeg', '.png','.ppm')):
            path = os.path.join(os.path.join(reference_dir,direc), file)
            img = cv2.imread(path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            # Preprocess image: Resize and normalize as the model expects
            # (Typical preprocessing includes resizing to 112x112 and normalization)
            img_resized = cv2.resize(img, (112, 112))
            img_norm = (img_resized / 127.5) - 1.0  # Normalize to [-1, 1]
            img_norm = np.transpose(img_norm, (2, 0, 1))  # HWC -> CHW
            img_norm = np.expand_dims(img_norm, axis=0)  # Add batch dimension
            img_norm = np.ascontiguousarray(img_norm, dtype=np.float32)
            
            # faces = model.forward(img)[0]
            faces = model.get_feat(img)[0]
            # faces = model.get(img)
            if len(faces) == 0:
                print(f"No face found in {file}")
                continue
            emb = faces
            embeddings.append(emb)
            reference_names.append(file)

allEmbeds = np.stack(embeddings)

# # Step 3: Process query image
# query_img = cv2.imread("output_unmask/00001/00001_930831_fa_a.jpg")
# query_faces = model.get(query_img)

# if len(query_faces) == 0:
#     print("No face found in query image.")
# else:
#     query_embedding = query_faces[0].embedding

#     # Step 4: Compare using cosine similarity
#     similarities = cosine_similarity([query_embedding], allEmbeds)[0]

#     # Step 5: Show results
#     best_idx = np.argmax(similarities)
#     best_match_name = reference_names[best_idx]
#     best_score = similarities[best_idx]

#     print(f"Best match: {best_match_name} (Score: {best_score:.4f})")

#     # Step 6: Show top-K matches
#     top_k = 5
#     top_indices = similarities.argsort()[::-1][:top_k]
#     print("\nTop matches:")
#     for i in top_indices:
#         print(f"{reference_names[i]} - Similarity: {similarities[i]:.4f}")


In [4]:
# # Step 3: Process query image
query_img = cv2.imread("output_texture/00001/00001_930831_fb_a.jpg")
query_faces = model.get_feat(query_img)[0]

if len(query_faces) == 0:
    print("No face found in query image.")
else:
    query_embedding = query_faces

    # Step 4: Compare using cosine similarity
    similarities = cosine_similarity([query_embedding], allEmbeds)[0]

    # Step 5: Show results
    best_idx = np.argmax(similarities)
    best_match_name = reference_names[best_idx]
    best_score = similarities[best_idx]

    print(f"Best match: {best_match_name} (Score: {best_score:.4f})")

    # Step 6: Show top-K matches
    top_k = 5
    top_indices = similarities.argsort()[::-1][:top_k]
    print("\nTop matches:")
    for i in top_indices:
        print(f"{reference_names[i]} - Similarity: {similarities[i]:.4f}")


Best match: 00180_931230_hr.ppm (Score: 0.9802)

Top matches:
00180_931230_hr.ppm - Similarity: 0.9802
00180_931230_hr.jpg - Similarity: 0.9794
00588_941031_hr.ppm - Similarity: 0.9790
00588_941031_hr.jpg - Similarity: 0.9785
00167_931230_pr.jpg - Similarity: 0.9784


In [4]:
!pip install onnxruntime

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 277.9 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 440.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 581.9 MB/s eta 0:00:00
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
